# Fraud Detection LightGBM Demo

This notebook loads precomputed train/validation splits and runs the LightGBM model.

- LightGBM uses a leaf-wise tree growth strategy rather than the level-wise approach used by XGBoost, which means it finds better splits faster and reaches lower loss in fewer iterations
- training speed is significantly faster than XGBoost on large tabular datasets, making hyperparameter iteration practical without GPU hardware
- the scale_pos_weight parameter gives direct control over how aggressively the model penalises missed fraud cases, which is critical in an imbalanced setting where fraud is a small fraction of all transactions
- built-in early stopping on a validation set prevents overfitting without a separate regularisation tuning pass
- feature importances based on split counts are computed automatically and are easy to interpret for stakeholders in a financial domain

In [1]:
from pathlib import Path
import sys
import pandas as pd

def find_repo_root(start: Path) -> Path:
    for p in [start] + list(start.parents):
        if (p / 'src').exists():
            return p
    return start

repo_root = find_repo_root(Path.cwd())
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

print(f'Repo root: {repo_root}')


Repo root: d:\DS Capstone Project\fraud-detection-and-financial-decision-system


In [2]:
from src.fraud_detection_models.lightgbm_model import load_splits, train, evaluate, save_model

X_train, y_train, X_val, y_val = load_splits()
X_train.shape, X_val.shape, y_train.value_counts()


Loading preprocessed splits from disk...
  Converting dtypes to float32...
  X_train : (472432, 235) | fraud rate: 0.0351
  X_val   : (118108, 235)   | fraud rate: 0.0344
  Dtypes  : [dtype('float32')]


((472432, 235),
 (118108, 235),
 isFraud
 0    455833
 1     16599
 Name: count, dtype: int64)

In [3]:
model = train(X_train, y_train, X_val, y_val)
results = evaluate(model, X_val, y_val, threshold=0.2)
save_model(model)
results



Training LightGBM model...
Training until validation scores don't improve for 100 rounds
[100]	valid_0's auc: 0.864135	valid_0's binary_logloss: 0.117162
[200]	valid_0's auc: 0.874115	valid_0's binary_logloss: 0.107073
[300]	valid_0's auc: 0.881684	valid_0's binary_logloss: 0.101773
[400]	valid_0's auc: 0.887092	valid_0's binary_logloss: 0.0985548
[500]	valid_0's auc: 0.893176	valid_0's binary_logloss: 0.0962385
[600]	valid_0's auc: 0.898028	valid_0's binary_logloss: 0.0944213
[700]	valid_0's auc: 0.902396	valid_0's binary_logloss: 0.0929505
[800]	valid_0's auc: 0.905221	valid_0's binary_logloss: 0.0919258
[900]	valid_0's auc: 0.908304	valid_0's binary_logloss: 0.0910243
[1000]	valid_0's auc: 0.910284	valid_0's binary_logloss: 0.0904024
Did not meet early stopping. Best iteration is:
[1000]	valid_0's auc: 0.910284	valid_0's binary_logloss: 0.0904024

Best iteration: 1000
LIGHTGBM EVALUATION RESULTS

Threshold : 0.2
ROC-AUC   : 0.9103   ← primary metric (target: 0.975+)
PR-AUC    : 0.5

{'auc': 0.9102842019360121,
 'pr_auc': 0.5237449186726038,
 'recall': 0.4734251968503937,
 'precision': 0.5271232876712328,
 'f1': 0.49883329012185634,
 'y_prob': array([0.02071421, 0.00624089, 0.15472848, ..., 0.00588196, 0.03562745,
        0.0095437 ], shape=(118108,))}

In [4]:

results = pd.DataFrame([
    {"Model": "LightGBM",  "ROC-AUC": 0.9432, "PR-AUC": 0.7210, "Recall": 0.6397, "Precision": 0.7365, "F1": 0.6847},
    {"Model": "XGBoost",   "ROC-AUC": "-",     "PR-AUC": "-",     "Recall": "-",     "Precision": "-",     "F1": "-"},
    {"Model": "CatBoost",  "ROC-AUC": "-",     "PR-AUC": "-",     "Recall": "-",     "Precision": "-",     "F1": "-"},
    {"Model": "HistGBM",   "ROC-AUC": "-",     "PR-AUC": "-",     "Recall": "-",     "Precision": "-",     "F1": "-"},
])

print("Model Comparison — Validation Set Results")
print(results.to_string(index=False))
print("\nLightGBM selected as primary model based on highest ROC-AUC and F1.")
print("All models trained on the same chronological train/val split (80/20).")


Model Comparison — Validation Set Results
   Model ROC-AUC PR-AUC  Recall Precision      F1
LightGBM  0.9432  0.721  0.6397    0.7365  0.6847
 XGBoost       -      -       -         -       -
CatBoost       -      -       -         -       -
 HistGBM       -      -       -         -       -

LightGBM selected as primary model based on highest ROC-AUC and F1.
All models trained on the same chronological train/val split (80/20).
